In [1]:
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
from selenium import webdriver
import time
import pandas as pd

In [4]:
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, TimeoutException, StaleElementReferenceException
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
import os


def close_modals(driver):
    """Aggressively close any open modal/dialog"""
    modal_xpaths = [
        '//*[@data-test="modal-close-btn"]',
        '//*[contains(@class,"modal_closeIcon")]',
        '//*[contains(@class,"CloseButton")]',
        '//button[contains(@aria-label,"Close")]',
        '//button[contains(@aria-label,"close")]',
        '//*[contains(@class,"SVGInline")][contains(@class,"close")]',
    ]
    for xpath in modal_xpaths:
        try:
            btn = driver.find_element(By.XPATH, xpath)
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(0.5)
            return True
        except NoSuchElementException:
            continue
    # Also try dismissing via JS if a dialog is open
    try:
        driver.execute_script("""
            var dialogs = document.querySelectorAll('dialog[open]');
            dialogs.forEach(function(d) { d.close(); d.remove(); });
            var overlays = document.querySelectorAll('[class*="modal"],[class*="Modal"],[class*="overlay"],[class*="Overlay"]');
            overlays.forEach(function(o) { o.remove(); });
        """)
        time.sleep(0.3)
    except:
        pass
    return False


def safe_click(driver, element):
    """Try normal click, fall back to JS click"""
    try:
        element.click()
    except (ElementClickInterceptedException, Exception):
        close_modals(driver)
        try:
            element.click()
        except:
            driver.execute_script("arguments[0].click();", element)


def get_text_by_xpath(driver, xpath):
    try:
        el = driver.find_element(By.XPATH, xpath)
        return el.text.strip() if el.text.strip() else -1
    except (NoSuchElementException, StaleElementReferenceException):
        return -1


def get_jobs(keyword, num_jobs, verbose):
    '''Gathers jobs as a dataframe, scraped from Glassdoor'''

    options = webdriver.ChromeOptions()
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    # options.add_argument('--headless')

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.set_window_size(1120, 1000)

    url = (
     f"https://www.glassdoor.com/Job/jobs.htm?sc.keyword={keyword}"
)
    driver.get(url)
    time.sleep(4)

    # Initial modal close attempt
    close_modals(driver)
    time.sleep(1)

    jobs = []

    while len(jobs) < num_jobs:

        # Always close modals at start of each page
        close_modals(driver)
        time.sleep(2)

        # Get job listing elements
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, '//li[@data-test="jobListing"]'))
            )
            job_listings = driver.find_elements(By.XPATH, '//li[@data-test="jobListing"]')
        except TimeoutException:
            print("Could not find job listings. Stopping.")
            break

        print(f"\n--- Page has {len(job_listings)} listings ---")

        for i in range(len(job_listings)):
            if len(jobs) >= num_jobs:
                break

            close_modals(driver)

            try:
                # Re-fetch to avoid stale elements
                job_listings = driver.find_elements(By.XPATH, '//li[@data-test="jobListing"]')
                job = job_listings[i]

                # Scroll into view then click
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", job)
                time.sleep(0.5)
                close_modals(driver)
                safe_click(driver, job)
                time.sleep(2)
                close_modals(driver)

                # --- Job Title --- Francisco
                job_title = -1
                for xpath in [
                    '//h1[contains(@class,"title")]',
                    '//h1[contains(@data-test,"jobTitle")]',
                    '//div[contains(@data-test,"job-title")]',
                    '//div[contains(@class,"jobTitle")]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        job_title = val
                        break

                # --- Company Name ---
                company_name = -1
                for xpath in [
                    '//div[@data-test="employer-name"]',
                    '//span[@data-test="employer-name"]',
                    '//div[contains(@class,"employerName")]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        company_name = val
                        break

                # --- Location ---
                location = -1
                for xpath in [
                    '//div[@data-test="location"]',
                    '//div[contains(@class,"location")]',
                    '//span[contains(@class,"location")]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        location = val
                        break

                # --- Job Description ---
                job_description = -1
                for xpath in [
                    '//div[@class="jobDescriptionContent desc"]',
                    '//div[contains(@class,"jobDescription")]',
                    '//div[@id="JobDescriptionContainer"]',
                    '//div[contains(@class,"JobDescription")]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        job_description = val
                        break

                # --- Salary ---
                salary_estimate = -1
                for xpath in [
                    '//span[@data-test="detailSalary"]',
                    '//div[contains(@class,"salary")]',
                    '//span[contains(@class,"salary")]',
                    '//div[@data-test="salary-estimate"]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        salary_estimate = val
                        break

                # --- Rating ---
                rating = -1
                for xpath in [
                    '//span[@data-test="detailRating"]',
                    '//span[contains(@class,"ratingNumber")]',
                    '//div[contains(@class,"ratingNumber")]',
                ]:
                    val = get_text_by_xpath(driver, xpath)
                    if val != -1:
                        rating = val
                        break

                # --- Company Overview Tab ---
                headquarters = size = founded = type_of_ownership = -1
                industry = sector = revenue = competitors = -1

                try:
                    overview_tab = None
                    for xpath in [
                        '//div[@data-tab-type="overview"]',
                        '//button[contains(text(),"Company")]',
                        '//a[contains(text(),"Company")]',
                    ]:
                        try:
                            overview_tab = driver.find_element(By.XPATH, xpath)
                            break
                        except NoSuchElementException:
                            continue

                    if overview_tab:
                        safe_click(driver, overview_tab)
                        time.sleep(1)

                        info_fields = {
                            "Headquarters": "headquarters",
                            "Size": "size",
                            "Founded": "founded",
                            "Type": "type_of_ownership",
                            "Industry": "industry",
                            "Sector": "sector",
                            "Revenue": "revenue",
                            "Competitors": "competitors",
                        }
                        results = {}
                        for label, varname in info_fields.items():
                            results[varname] = get_text_by_xpath(
                                driver,
                                f'.//div[@class="infoEntity"]//label[text()="{label}"]//following-sibling::*'
                            )

                        headquarters     = results.get("headquarters", -1)
                        size             = results.get("size", -1)
                        founded          = results.get("founded", -1)
                        type_of_ownership = results.get("type_of_ownership", -1)
                        industry         = results.get("industry", -1)
                        sector           = results.get("sector", -1)
                        revenue          = results.get("revenue", -1)
                        competitors      = results.get("competitors", -1)

                except Exception as e:
                    if verbose:
                        print(f"  Overview tab error: {e}")

                jobs.append({
                    "Job Title": job_title,
                    "Salary Estimate": salary_estimate,
                    "Job Description": job_description,
                    "Rating": rating,
                    "Company Name": company_name,
                    "Location": location,
                    "Headquarters": headquarters,
                    "Size": size,
                    "Founded": founded,
                    "Type of ownership": type_of_ownership,
                    "Industry": industry,
                    "Sector": sector,
                    "Revenue": revenue,
                    "Competitors": competitors
                })

                print(f"Progress: {len(jobs)}/{num_jobs} | {job_title} @ {company_name} | {location}")

                if verbose:
                    print(f"  Salary: {salary_estimate} | Rating: {rating}")

            except StaleElementReferenceException:
                print(f"Stale element on job {i}, skipping.")
                continue
            except Exception as e:
                print(f"Error on job {i}: {e}")
                continue

        # --- Next Page ---
        try:
            next_btn = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH,
                    '//button[@data-test="pagination-next"] | //a[@data-test="pagination-next"]'
                ))
            )
            close_modals(driver)
            safe_click(driver, next_btn)
            time.sleep(3)
        except (NoSuchElementException, TimeoutException):
            print(f"No next page. Scraping complete. Got {len(jobs)} jobs.")
            break

    driver.quit()
    return pd.DataFrame(jobs)


# --- Run ---

keywords = [
    "data scientist",
    "data analyst",
    "machine learning engineer",
    "data engineer",
    "business analyst",
    "analytics engineer",
    "AI engineer",
    "research scientist"
]
all_jobs = []

for kw in keywords:
    df = get_jobs(kw, 5000, False)
    all_jobs.append(df)

final_df = pd.concat(all_jobs, ignore_index=True)
final_df.drop_duplicates(inplace=True)
os.makedirs("output", exist_ok=True)
final_df.to_csv("output/glassdoor_jobs.csv", index=False)
print(f"\nShape: {final_df.shape}")
print(final_df[["Job Title", "Company Name", "Location", "Salary Estimate", "Rating"]].head(10))


--- Page has 30 listings ---
Progress: 1/5000 | -1 @ Novartis
4.0 | India
Progress: 2/5000 | -1 @ Novartis
4.0 | Chennai
Progress: 3/5000 | -1 @ Novartis
4.0 | Telangana
Progress: 4/5000 | -1 @ Novartis
4.0 | Hyderābād
Progress: 5/5000 | -1 @ Novartis
4.0 | Gurgaon
Progress: 6/5000 | -1 @ Novartis
4.0 | Gurgaon
Progress: 7/5000 | -1 @ Novartis
4.0 | Chennai
Progress: 8/5000 | -1 @ Novartis
4.0 | Noida
Progress: 9/5000 | -1 @ Novartis
4.0 | Noida
Progress: 10/5000 | -1 @ Novartis
4.0 | Noida
Progress: 11/5000 | -1 @ Novartis
4.0 | Noida
Progress: 12/5000 | -1 @ Novartis
4.0 | India
Progress: 13/5000 | -1 @ Novartis
4.0 | Gurgaon
Progress: 14/5000 | -1 @ Novartis
4.0 | India
Progress: 15/5000 | -1 @ Novartis
4.0 | Noida
Progress: 16/5000 | -1 @ Novartis
4.0 | Hyderābād
Progress: 17/5000 | -1 @ Novartis
4.0 | Amarāvati
Progress: 18/5000 | -1 @ Novartis
4.0 | Gurgaon
Progress: 19/5000 | -1 @ Novartis
4.0 | India
Progress: 20/5000 | -1 @ Novartis
4.0 | Hyderābād
Progress: 21/5000 | -1 @ No